# TWO WORLDS — Kaggle T4×2 Russian Video Studio v3
Private 6-story production. **START WITH CPU PREFLIGHT**, without GPU quota or model downloads.
Select a PRIVATE Kaggle dataset containing `Two_Worlds_Studio_RU_Batch03_Ready_For_Production.zip`. Don't share private scripts.
After successful preflight, select GPU T4 x2 in Kaggle Notebook → Settings → Accelerator and change `PHASE` to image/voice. Kaggle has limited free quota; GPU sessions are not unlimited.


In [ ]:
# PARAMETERS — this cell should be edited only after a successful preflight.
STORY = "13"                 # 13..18
PHASE = "preflight"          # preflight / image / voice / music / render
FIRST_SCENE = 1
MAX_SCENES = 1              # test one scene first
GPU_INDEX = 0               # GPU T4x2: two independent 16GB cards
MODEL_DIR = "/kaggle/input/two-worlds-models"
PRIVATE_SOURCE = None       # set full /kaggle/input/...zip if auto-discovery ambiguous
PRIVATE_RECOVERY = None     # optional previous PRIVATE Kaggle ZIP input
SPEAKER = "Serena"
TIME_BUDGET = 2400
INSTALL_DEPENDENCIES = False  # True only with Kaggle Internet enabled
BOOTSTRAP_WEIGHTS = False     # True downloads huge model weights; uses time/space


In [ ]:
import glob, os, subprocess, sys
from pathlib import Path
if PRIVATE_SOURCE is None:
    matches = sorted(glob.glob('/kaggle/input/**/Two_Worlds_Studio_RU_Batch03_Ready_For_Production.zip',recursive=True))
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly ONE private Batch03 input ZIP. Found {matches}. Attach a PRIVATE Kaggle Dataset.')
    PRIVATE_SOURCE = matches[0]
if PHASE in ('image','voice'):
    os.environ['CUDA_VISIBLE_DEVICES'] = str(GPU_INDEX)
if BOOTSTRAP_WEIGHTS:
    if PHASE not in ('image','voice') or not INSTALL_DEPENDENCIES:
        raise RuntimeError('Bootstrap only with explicit generation phase + dependencies')
    MODEL_DIR = '/kaggle/working/private_models'
if INSTALL_DEPENDENCIES and PHASE in ('image','voice'):
    packages = (['qwen-tts','soundfile'] if PHASE=='voice' else
                ['diffusers','transformers','accelerate','safetensors'])
    subprocess.run([sys.executable,'-m','pip','install','-q',*packages],check=True)
    if BOOTSTRAP_WEIGHTS:
        from huggingface_hub import snapshot_download
        model = ('black-forest-labs/FLUX.2-klein-4B' if PHASE=='image'
                 else 'Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice')
        snapshot_download(model,local_dir=f'{MODEL_DIR}/{PHASE}')
print('PRIVATE source:',PRIVATE_SOURCE)
print('PHASE:',PHASE,'STORY:',STORY,'SCENES:',FIRST_SCENE,MAX_SCENES)


In [ ]:
# Public SOURCE CODE only. Private texts and generated media stay in Kaggle private input/output.
repo=Path('/kaggle/working/two_worlds_studio')
if not repo.exists():
    subprocess.run(['git','clone','--quiet','--depth','1','--branch',
        'feat/kaggle-t4x2-russian-short-factory-20261030',
        'https://github.com/19wolfalone92/Two-Worlds-Studio.git',str(repo)],check=True)
sys.path.insert(0,str(repo))
from studio.kaggle_shorts_worker import prepare_input, run_job, export_private_assets
work=prepare_input(Path(PRIVATE_SOURCE),Path('/kaggle/working/private_work_'+STORY),STORY,
                   recovery=Path(PRIVATE_RECOVERY) if PRIVATE_RECOVERY else None)
print('SOURCE SHA-256 preserved. Prepared:',work)


In [ ]:
# Runs only the selected bounded phase. Preflight mode does NOT request GPU.
out=Path('/kaggle/working/private_result_'+STORY+'_'+PHASE)
r=run_job(work,out,Path(MODEL_DIR),story=STORY,phase=PHASE,
          first_scene=FIRST_SCENE,max_scenes=MAX_SCENES,gpu_id=0,
          time_budget_s=TIME_BUDGET,speaker=SPEAKER)
result_zip=Path('/kaggle/working/TwoWorlds_PRIVATE_'+STORY+'_'+PHASE+'.zip')
package=export_private_assets(work,result_zip,r)
print('PHASE STATUS:',r['status'],'
GENERATED:',len(r['made']))
print('Private ZIP:',package['zip'],'SHA-256:',package['sha256'])
print('Human artistic review: REQUIRED. Never auto-publish.')
